In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import zipfile

zip_path = "/content/drive/MyDrive/VIP_Midterm/CellMob_AUB.zip"
extract_path = "/content/CellMob_AUB"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully.")

Dataset extracted successfully.


In [ ]:
import os
import pandas as pd

base_dirs = [
    "/content/CellMob_AUB/CellMob_AUB/CellMob",
    "/content/CellMob_AUB/CellMob_AUB/KAUST_additional_modes"
]

rows = []

for base_dir in base_dirs:
    for folder in os.listdir(base_dir):
        folder_path = os.path.join(base_dir, folder)
        if not os.path.isdir(folder_path):
            continue

        if base_dir.endswith("CellMob"):
            parts = folder.split("_")
            label = parts[0]
            city = parts[-1]
        else:
            label = folder
            city = "kaust_extra"

        for file in os.listdir(folder_path):
            if file.endswith(".csv"):
                rows.append({
                    "folder": folder,
                    "file": file,
                    "city": city,
                    "label": label,
                    "path": os.path.join(folder_path, file)
                })

df_index = pd.DataFrame(rows)
print("Number of CSV files:", len(df_index))

Number of CSV files: 228


In [ ]:
from collections import Counter

column_counter = Counter()

for path in df_index["path"]:
    df_tmp = pd.read_csv(path, nrows=1, low_memory=False)
    for c in df_tmp.columns:
        column_counter[c] += 1

df_columns = pd.DataFrame({
    "column": list(column_counter.keys()),
    "appears_in_n_files": list(column_counter.values())
}).sort_values(by=["appears_in_n_files", "column"], ascending=[False, True])

all_file_cols = df_columns[df_columns["appears_in_n_files"] == len(df_index)]
common_columns = all_file_cols["column"].tolist()

print("Columns appearing in all files:", len(common_columns))

Columns appearing in all files: 43


In [ ]:
clean_dfs = []

for _, row in df_index.iterrows():
    df = pd.read_csv(row["path"], low_memory=False)
    df = df[common_columns].copy()
    df["city"] = row["city"]
    df["label"] = row["label"]
    df["source_file"] = row["file"]
    clean_dfs.append(df)

df_all = pd.concat(clean_dfs, ignore_index=True)

print("Combined shape:", df_all.shape)
print(df_all["city"].value_counts())
print(df_all["label"].value_counts())

Combined shape: (6187368, 46)
city
kaust_extra    2103892
kaust          1832613
mekkah         1191474
jeddah          903774
kz              155615
Name: count, dtype: int64
label
walk            2042271
car             1179959
scooter          860708
bus              766555
bike             592601
jog              285524
bus_ondemand     236663
motorcycle       128396
train             94691
Name: count, dtype: int64


In [ ]:
feature_cols = [c for c in df_all.columns if c not in ["city", "label", "source_file"]]

for col in feature_cols:
    df_all[col] = pd.to_numeric(df_all[col], errors="coerce")

df_all[feature_cols] = df_all[feature_cols].fillna(0)

print("Remaining NaNs:", df_all[feature_cols].isna().sum().sum())
print("Number of feature columns:", len(feature_cols))

Remaining NaNs: 0
Number of feature columns: 43


In [ ]:
from sklearn.preprocessing import LabelEncoder

df_all_4 = df_all[df_all["label"].isin(["bus", "car", "train", "walk"])].copy()

le4 = LabelEncoder()
df_all_4["label_encoded"] = le4.fit_transform(df_all_4["label"])

print("4-class shape:", df_all_4.shape)
print(dict(zip(le4.classes_, le4.transform(le4.classes_))))
print(df_all_4["label"].value_counts())

4-class shape: (4083476, 47)
{'bus': np.int64(0), 'car': np.int64(1), 'train': np.int64(2), 'walk': np.int64(3)}
label
walk     2042271
car      1179959
bus       766555
train      94691
Name: count, dtype: int64


In [ ]:
import numpy as np

client_samples_4 = {}
max_per_client = 100000

for city in df_all_4["city"].unique():
    df_city = df_all_4[df_all_4["city"] == city]

    if len(df_city) > max_per_client:
        df_city = df_city.sample(n=max_per_client, random_state=42)

    X_city = df_city[feature_cols].values.astype("float32")
    y_city = df_city["label_encoded"].values.astype("int64")

    client_samples_4[city] = (X_city, y_city)

for city in client_samples_4:
    print(city, client_samples_4[city][0].shape, sorted(set(client_samples_4[city][1])))

mekkah (100000, 43) [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
kz (100000, 43) [np.int64(1), np.int64(3)]
kaust (100000, 43) [np.int64(0), np.int64(1), np.int64(3)]
jeddah (100000, 43) [np.int64(0), np.int64(1), np.int64(3)]


In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

batch_size = 256
client_loaders_4 = {}

for city, (X_city, y_city) in client_samples_4.items():
    X_tensor = torch.tensor(X_city, dtype=torch.float32)
    y_tensor = torch.tensor(y_city, dtype=torch.long)

    dataset_city = TensorDataset(X_tensor, y_tensor)
    loader = DataLoader(dataset_city, batch_size=batch_size, shuffle=True)

    client_loaders_4[city] = loader

df_test_4 = df_all_4.sample(n=40000, random_state=42)

X_test_fl_4 = torch.tensor(df_test_4[feature_cols].values.astype("float32"), dtype=torch.float32)
y_test_fl_4 = torch.tensor(df_test_4["label_encoded"].values.astype("int64"), dtype=torch.long)

print("X_test_fl_4 shape:", X_test_fl_4.shape)
print("y_test_fl_4 shape:", y_test_fl_4.shape)

X_test_fl_4 shape: torch.Size([40000, 43])
y_test_fl_4 shape: torch.Size([40000])


In [ ]:
import torch.nn as nn
import torch.optim as optim
from copy import deepcopy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

class TMDNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_classes=4):
        super(TMDNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.net(x)

def evaluate_model(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        outputs = model(X_test.to(device))
        preds = torch.argmax(outputs, dim=1).cpu()
        accuracy = (preds == y_test.cpu()).float().mean().item()
    return accuracy

def clone_state_dict(state_dict):
    return {k: v.detach().clone().to(device) for k, v in state_dict.items()}

def zero_state_dict_like(state_dict):
    return {k: torch.zeros_like(v).to(device) for k, v in state_dict.items()}

def average_model_states(local_states, client_sizes):
    total_size = sum(client_sizes)
    avg_state = {}

    for key in local_states[0].keys():
        avg_state[key] = sum(
            (client_sizes[i] / total_size) * local_states[i][key]
            for i in range(len(local_states))
        )

    return avg_state

def train_local_fedyogi(model, loader, epochs=1, lr=0.001):
    model = deepcopy(model).to(device)
    model.train()

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)

    for _ in range(epochs):
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

    return clone_state_dict(model.state_dict())

def fedyogi_server_update(global_state, avg_local_state, m, v, t, lr=0.01, beta1=0.9, beta2=0.99, eps=1e-8):
    new_global = {}

    for key in global_state.keys():
        delta = avg_local_state[key] - global_state[key]

        m[key] = beta1 * m[key] + (1 - beta1) * delta
        v[key] = v[key] - (1 - beta2) * (delta * delta) * torch.sign(v[key] - (delta * delta))

        m_hat = m[key] / (1 - beta1 ** t)
        v_hat = v[key] / (1 - beta2 ** t)

        new_global[key] = global_state[key] + lr * m_hat / (torch.sqrt(torch.abs(v_hat)) + eps)

    return new_global, m, v

Using device: cpu


In [ ]:
fedyogi_model_4 = TMDNet(input_dim=len(feature_cols), hidden_dim=64, num_classes=4).to(device)

global_state_4 = clone_state_dict(fedyogi_model_4.state_dict())
m_4 = zero_state_dict_like(global_state_4)
v_4 = zero_state_dict_like(global_state_4)

num_rounds_fedyogi = 10
local_epochs_fedyogi = 1
lr_local = 0.001
lr_server = 0.01

fedyogi_accuracies_4class = []

for rnd in range(num_rounds_fedyogi):
    local_states = []
    client_sizes = []

    fedyogi_model_4.load_state_dict(global_state_4)

    for city, loader in client_loaders_4.items():
        local_state = train_local_fedyogi(
            fedyogi_model_4,
            loader,
            epochs=local_epochs_fedyogi,
            lr=lr_local
        )
        local_states.append(local_state)
        client_sizes.append(len(loader.dataset))

    avg_local_state = average_model_states(local_states, client_sizes)

    global_state_4, m_4, v_4 = fedyogi_server_update(
        global_state_4,
        avg_local_state,
        m_4,
        v_4,
        t=rnd + 1,
        lr=lr_server
    )

    fedyogi_model_4.load_state_dict(global_state_4)

    print(f"Completed 4-class FedYogi round {rnd+1}/{num_rounds_fedyogi}")
    acc = evaluate_model(fedyogi_model_4, X_test_fl_4, y_test_fl_4)
    fedyogi_accuracies_4class.append(acc)

print("FedYogi accuracies:", fedyogi_accuracies_4class)
print("FedYogi best:", max(fedyogi_accuracies_4class))
print("FedYogi final:", fedyogi_accuracies_4class[-1])

Completed 4-class FedYogi round 1/10
Completed 4-class FedYogi round 2/10
Completed 4-class FedYogi round 3/10
Completed 4-class FedYogi round 4/10
Completed 4-class FedYogi round 5/10
Completed 4-class FedYogi round 6/10
Completed 4-class FedYogi round 7/10
Completed 4-class FedYogi round 8/10
Completed 4-class FedYogi round 9/10
Completed 4-class FedYogi round 10/10
FedYogi accuracies: [0.3807249963283539, 0.3725000023841858, 0.35327500104904175, 0.32284998893737793, 0.3393000066280365, 0.42602500319480896, 0.5166500210762024, 0.5138999819755554, 0.5149750113487244, 0.51357501745224]
FedYogi best: 0.5166500210762024
FedYogi final: 0.51357501745224


In [ ]:
print("Centralized accuracy: 0.8889")
print("FedAvg best accuracy: 0.5123")
print("FedAvg final accuracy: 0.4832")
print("FedAdam best accuracy: 0.5363")
print("FedAdam final accuracy: 0.4579")
print("FedAdagrad best accuracy: 0.5188")
print("FedAdagrad final accuracy: 0.5188")
print("FedYogi best accuracy:", max(fedyogi_accuracies_4class))
print("FedYogi final accuracy:", fedyogi_accuracies_4class[-1])

Centralized accuracy: 0.8889
FedAvg best accuracy: 0.5123
FedAvg final accuracy: 0.4832
FedAdam best accuracy: 0.5363
FedAdam final accuracy: 0.4579
FedAdagrad best accuracy: 0.5188
FedAdagrad final accuracy: 0.5188
FedYogi best accuracy: 0.44817501306533813
FedYogi final accuracy: 0.44817501306533813
